# MH-04. Ready-to-run build

Processing on the raw superset:
- Lookback filter: years_in_ehr_before_index >= 1.0 (the SOLE filter).
- Node floor -> V: keep standalone nodes with >= 200 case-persons; pool the rest by 3-digit
  category (positive if rule-of-two on any child) and keep pooled node if >= 200; 295.x
  force-retained regardless.
- X coverage floor: drop features with < 5% observed over the lookback cohort.
- Reduction (X only): median-fill -> PCA on dense numeric block. days_since columns are
  dropped from the reducer input. Interpretable block + obs-intensity + mask columns kept raw
  outside PCA, then concatenated back. (No FAMD/MFA.)


## 1. Load + lookback filter

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = Path('output/mh')
LOOKBACK_YEARS = None         # None = no lookback filter (use full cohort)
NODE_FLOOR = 200
COVERAGE_FLOOR = 0.01
PSYCHOSIS_PARENT = 295
PCA_VARIANCE = 0.90           # keep fewest PCs reaching this cumulative explained variance

spine = pd.read_parquet(OUTPUT_DIR / 'spine.parquet')
Y = pd.read_parquet(OUTPUT_DIR / 'Y_nodes.parquet').set_index('person_id')
R = pd.read_parquet(OUTPUT_DIR / 'R_mask.parquet').set_index('person_id')
X = pd.read_parquet(OUTPUT_DIR / 'X_raw.parquet').set_index('person_id')
node_table = pd.read_parquet(OUTPUT_DIR / 'mh_node_table.parquet')

# No lookback filter: use the full cohort so X coverage is computed over everyone.
if LOOKBACK_YEARS is None:
    keep = sorted(spine['person_id'].tolist())
    print(f'no lookback filter: keeping full cohort n={len(keep)}')
else:
    keep = sorted(spine[spine['years_in_ehr_before_index'] >= LOOKBACK_YEARS]['person_id'].tolist())
    print(f'lookback >= {LOOKBACK_YEARS}y: {len(spine)} -> {len(keep)}')

spine_f = spine[spine['person_id'].isin(keep)].sort_values('person_id').reset_index(drop=True)
Y, R, X = Y.reindex(keep), R.reindex(keep), X.reindex(keep)


## 2. Node floor -> V (standalone >=200, pool rest, 295.x force-retained)

In [ ]:
parent_of = dict(zip(node_table['phecode'], node_table['parent3']))
def case_persons(col):   # masked cells excluded (NaN), count Y==1
    return int(np.nansum(col.values == 1))

standalone = [n for n in Y.columns if case_persons(Y[n]) >= NODE_FLOOR]
force = [n for n in Y.columns if parent_of.get(n) == PSYCHOSIS_PARENT]
standalone = sorted(set(standalone) | set(force), key=lambda s: float(s))

remaining = [n for n in Y.columns if n not in standalone]
pooled_cols, pool_meta = {}, []
from collections import defaultdict
by_parent = defaultdict(list)
for n in remaining:
    by_parent[parent_of.get(n)].append(n)
for p3, kids in by_parent.items():
    pooled = Y[kids].max(axis=1, skipna=True)   # positive if any child positive
    if case_persons(pooled.to_frame('p')['p']) >= NODE_FLOOR:
        name = f'POOL_{int(p3)}'
        pooled_cols[name] = pooled
        pool_meta.append({'pooled_node': name, 'parent3': p3, 'children': kids})

V_standalone = Y[standalone]
V_pooled = pd.DataFrame(pooled_cols, index=Y.index) if pooled_cols else pd.DataFrame(index=Y.index)
Y_V = pd.concat([V_standalone, V_pooled], axis=1)
R_V = (~np.isnan(Y_V.values)).astype('int8')
R_V = pd.DataFrame(R_V, index=Y_V.index, columns=Y_V.columns)
print(f'|V| = {Y_V.shape[1]} (standalone {len(standalone)} + pooled {len(pooled_cols)})')


## 3. X coverage floor (>= 5% observed)

In [ ]:
raw_cols  = [c for c in X.columns if c.endswith('__raw')]
mask_cols = [c for c in X.columns if c.endswith('__mask')]
ds_cols   = [c for c in X.columns if c.endswith('__days_since')]

# Coverage from mask where a mask exists. A raw column WITHOUT a mask (e.g. ObsIntensity counts)
# is fully observed -> treated as 100% coverage and always kept.
cov = X[mask_cols].mean() if mask_cols else pd.Series(dtype=float)

def base_of(raw_col):
    return raw_col[:-len('__raw')]

kept_raw = []
for rc in raw_cols:
    m = base_of(rc) + '__mask'
    if m in cov.index:
        if cov[m] >= COVERAGE_FLOOR:
            kept_raw.append(rc)
    else:
        kept_raw.append(rc)  # no mask -> fully observed, keep

kept_mask = [base_of(rc) + '__mask' for rc in kept_raw if base_of(rc) + '__mask' in X.columns]
print(f'raw features kept after {COVERAGE_FLOOR:.0%} coverage gate: {len(kept_raw)} / {len(raw_cols)}')
print('mask feature columns kept:', len(kept_mask), '| days_since dropped from reducer:', len(ds_cols))


## 4. Split interpretable-raw vs reduced, median-fill -> PCA

In [ ]:
# "All into PCA": every kept raw column enters the reducer. Only observed_mask columns are kept
# raw as standalone features (missingness is informative); days_since is excluded.
INTERPRETABLE_PREFIXES = ()      # keep nothing raw outside the reducer (all numeric raw -> PCA)
interp_raw = [c for c in kept_raw if INTERPRETABLE_PREFIXES and c.startswith(INTERPRETABLE_PREFIXES)]
reduce_raw = [c for c in kept_raw if c not in interp_raw]
print('columns entering PCA (reduce_raw):', len(reduce_raw))


### Heavy step: median-fill -> standardize -> PCA(0.90)

In [ ]:
# Simple, fast missing-value fill: per-column MEDIAN, then standardize, then PCA(0.90).
# (Chosen over median-fill for speed/simplicity; most reduce_raw cols are sparse, so values
# are largely median-filled. Recorded in provenance.)
if reduce_raw:
    Xr = X[reduce_raw].astype('float64')
    imputed_share = float(1.0 - Xr.notna().values.mean())
    Xi = SimpleImputer(strategy='median').fit_transform(Xr.values)
    Xi = StandardScaler().fit_transform(Xi).astype('float32')
    pca = PCA(n_components=PCA_VARIANCE, svd_solver='full')
    scores = pca.fit_transform(Xi).astype('float32')
    k = scores.shape[1]
    reduced = pd.DataFrame(scores, index=X.index, columns=[f'PC{i:03d}' for i in range(1, k+1)])
    print(f'reduced block: {len(reduce_raw)} raw feats -> {k} PCs '
          f'(cum var {pca.explained_variance_ratio_.sum():.4f} >= {PCA_VARIANCE})')
    print(f'median-imputed cell share in reducer input: {imputed_share:.1%}')
else:
    reduced = pd.DataFrame(index=X.index); imputed_share = 0.0
    print('no dense block to reduce')


## 5. Assemble ready-to-run X, save aligned build

In [ ]:
X_ready = pd.concat([X[interp_raw], X[kept_mask], reduced], axis=1)
X_ready = X_ready.reindex(sorted(keep))
assert list(X_ready.index) == sorted(keep) == list(Y_V.reindex(sorted(keep)).index)

BUILD = OUTPUT_DIR / 'ready_to_run'
BUILD.mkdir(exist_ok=True)
spine_f.to_parquet(BUILD / 'spine.parquet', index=False)
Y_V.reindex(sorted(keep)).reset_index().to_parquet(BUILD / 'Y.parquet', index=False)
R_V.reindex(sorted(keep)).reset_index().to_parquet(BUILD / 'R.parquet', index=False)
X_ready.reset_index().rename(columns={'index': 'person_id'}).to_parquet(BUILD / 'X.parquet', index=False)

manifest = pd.DataFrame(
    [{'node': n, 'kind': 'standalone'} for n in standalone] +
    [{'node': m['pooled_node'], 'kind': 'pooled', 'children': ';'.join(m['children'])} for m in pool_meta]
)
manifest.to_csv(BUILD / 'V_manifest.csv', index=False)

pd.DataFrame({'x_feature': interp_raw + kept_mask + list(reduced.columns),
             'kept_as': ['raw'] * len(interp_raw) + ['mask'] * len(kept_mask) + ['pca'] * reduced.shape[1]}
            ).to_csv(BUILD / 'X_manifest.csv', index=False)

prov = {
    'coverage_floor': COVERAGE_FLOOR,
    'pca_variance_target': PCA_VARIANCE,
    'lookback_filter': LOOKBACK_YEARS,
    'imputation': 'median',
    'n_reduce_features': len(reduce_raw),
    'n_pcs': int(reduced.shape[1]),
    'imputed_cell_share_in_reducer': round(float(imputed_share), 4),
    'note': 'No lookback filter; X PCs from sparse features, values largely median-imputed.',
}
(BUILD / 'X_provenance.json').write_text(json.dumps(prov, indent=2))

print('ready-to-run X:', X_ready.shape, '| Y_V:', Y_V.shape)
print('provenance:', prov)
print('saved ->', BUILD.resolve())
